# 01 — Data Exploration

Sanity-check all input datasets before analysis:
- WBGT daily GeoTIFFs (CHIRTS-ERA5)
- Population raster (1 km)
- Relative Wealth Index raster (2.4 km)
- Country boundaries

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA = ROOT / 'data'
print('Project root:', ROOT)

## 1. WBGT — check a sample file

In [ ]:
# List available WBGT files
wbgt_dir = DATA / 'wbgt'
wbgt_files = sorted(wbgt_dir.glob('WBGT.*.tif'))
print(f'WBGT files found: {len(wbgt_files)}')
if wbgt_files:
    print('First:', wbgt_files[0].name)
    print('Last: ', wbgt_files[-1].name)

In [ ]:
# Inspect one file
sample_file = wbgt_files[0] if wbgt_files else None

if sample_file:
    with rasterio.open(sample_file) as src:
        print('CRS:       ', src.crs)
        print('Resolution:', src.res)
        print('Shape:     ', src.shape)
        print('Bounds:    ', src.bounds)
        print('Nodata:    ', src.nodata)
        arr = src.read(1).astype(float)
        arr[arr == src.nodata] = np.nan

    print(f'\nValue range: {np.nanmin(arr):.1f} - {np.nanmax(arr):.1f} °C')

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(arr, cmap='RdYlBu_r', vmin=0, vmax=40)
    plt.colorbar(im, ax=ax, label='WBGT (°C)')
    ax.set_title(f'WBGT — {sample_file.name}')
    plt.tight_layout()
    plt.show()

## 2. Population raster

In [ ]:
pop_path = DATA / config['data']['population']

with rasterio.open(pop_path) as src:
    print('CRS:       ', src.crs)
    print('Resolution:', src.res)
    print('Shape:     ', src.shape)
    print('Bounds:    ', src.bounds)
    print('Nodata:    ', src.nodata)
    # Read a small overview to avoid loading full raster
    pop_overview = src.read(1, out_shape=(1, 500, 1000), resampling=rasterio.enums.Resampling.average)[0]

pop_overview = pop_overview.astype(float)
pop_overview[pop_overview < 0] = np.nan
print(f'\nTotal population (approx): {np.nansum(pop_overview):,.0f}')

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(np.log1p(pop_overview), cmap='viridis')
plt.colorbar(im, ax=ax, label='log(population + 1)')
ax.set_title('Population (overview)')
plt.tight_layout()
plt.show()

## 3. Relative Wealth Index

In [ ]:
rwi_path = DATA / config['data']['rwi']

with rasterio.open(rwi_path) as src:
    print('CRS:       ', src.crs)
    print('Resolution:', src.res)
    print('Shape:     ', src.shape)
    print('Bounds:    ', src.bounds)
    print('Nodata:    ', src.nodata)
    rwi_overview = src.read(1, out_shape=(1, 500, 1000), resampling=rasterio.enums.Resampling.average)[0]

rwi_overview = rwi_overview.astype(float)
rwi_overview[rwi_overview == -999] = np.nan
print(f'\nRWI range: {np.nanmin(rwi_overview):.2f} to {np.nanmax(rwi_overview):.2f}')

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(rwi_overview, cmap='RdYlGn', vmin=-2, vmax=2)
plt.colorbar(im, ax=ax, label='RWI')
ax.set_title('Relative Wealth Index (overview)')
plt.tight_layout()
plt.show()

## 4. Country boundaries

In [ ]:
boundaries_path = DATA / config['data']['boundaries']
iso_codes = config['iso_codes']

world = gpd.read_file(boundaries_path)
print('Boundary file columns:', list(world.columns))
print('Total countries:', len(world))

# Identify which ISO column to use
for col in ['ADM0_A3', 'ISO_A3', 'iso_a3', 'GID_0']:
    if col in world.columns:
        iso_col = col
        break

lmics = world[world[iso_col].isin(iso_codes)]
print(f'LMICs matched: {len(lmics)} / {len(iso_codes)}')

missing = set(iso_codes) - set(lmics[iso_col])
if missing:
    print('Missing:', sorted(missing))

fig, ax = plt.subplots(figsize=(14, 7))
world.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.3)
lmics.plot(ax=ax, color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_title('92 LMICs in analysis')
ax.axis('off')
plt.tight_layout()
plt.show()